# Research Scratchpad

Use the same plumbing as the rest of the project for quick analyses.

In [1]:
import os
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import load_config
from src.utils.logger import get_logger
from src.data.data_manager import DataManager

config = load_config()
logger = get_logger("notebooks.research")
data_manager = DataManager(config=config)


Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
2025-11-18 19:29:32,241 | INFO | src.data.market_data | Market Data handler initialized | base_url: https://data.alpaca.markets


## Load whatever dataset you need

In [2]:
df = data_manager.load_data('engineered_features')
if df.empty:
    raise RuntimeError("Run 02_feature_engineering notebook first to generate data with features.")

symbol = config['trading']['symbol']
df.tail()

2025-11-18 19:29:34,235 | INFO | src.data.data_manager | Loading dataset engineered_features from data/processed/engineered_features.parquet


,open,high,low,close,volume,sma_20,sma_50,rsi_14,return_1d,lag_close_1,day_of_week
t,,,,,,,,,,,
2024-05-17 04:00:00+00:00,943.69,947.4,918.06,924.79,35989353,879.3725,882.4177,59.353459,-0.019924,943.59,4
2024-05-20 04:00:00+00:00,937.50,952.0,934.40,947.80,31876446,887.0035,883.8681,65.995876,0.024881,924.79,0
2024-05-21 04:00:00+00:00,935.99,954.0,931.80,953.86,32894646,893.4850,885.7905,76.341057,0.006394,947.80,1
2024-05-22 04:00:00+00:00,954.59,960.2,932.49,949.50,54865849,901.1215,886.3979,71.649362,-0.004571,953.86,2
2024-05-23 04:00:00+00:00,1020.28,1063.2,1015.20,1037.99,83506528,911.7050,888.9801,77.827215,0.093196,949.50,3


## Quick correlatioj peek

In [3]:
fast = config['backtest']['strategy_params'].get('fast_period', 20)
slow = config['backtest']['strategy_params'].get('slow_period', 50)
cols = ['close', f'sma_{fast}', f'sma_{slow}', 'rsi_14', 'return_1d']
cols = [col for col in cols if col in df.columns]

if cols:
    display(df[cols].corr())
else:
    print("No specified columns found for correlation analysis.")
    display(df.head())

,close,sma_20,sma_50,rsi_14,return_1d
close,1.000000,0.939682,0.855869,-0.689067,-0.029986
sma_20,0.939682,1.000000,0.936655,-0.866713,-0.166603
sma_50,0.855869,0.936655,1.000000,-0.817372,-0.131840
rsi_14,-0.689067,-0.866713,-0.817372,1.000000,0.299949
return_1d,-0.029986,-0.166603,-0.131840,0.299949,1.000000
